At a minimum it would be good to analyse the changing shape of the network over time, the changing shape of the Giant Component, the number of isolates, etc (subfigures to the right as linegraphs of counts of these two types of component).

In [1]:
import os
import ast
import pandas as pd
import numpy as np
from collections import defaultdict
import ast, functools
from typing import Optional, List
from tqdm.notebook import tqdm
tqdm.pandas()

df_all = pd.read_excel('../data/df_dimensions.xlsx', index_col=0)

In [2]:
df = df_all[['id','researchers','research_orgs','year']].copy()

In [3]:
# Function to safely parse 'researchers' field with multiple repair attempts
import ast, json, re
import pandas as pd
from tqdm import tqdm

def safe_parse_researchers(s):
    """
    Return a list parsed from string s with multiple fallback/repair attempts.
    """
    if pd.isna(s):
        return []
    # If it's already a list/tuple, return list
    if isinstance(s, (list, tuple)):
        return list(s)

    orig = str(s)
    # 1) try ast.literal_eval directly
    try:
        val = ast.literal_eval(orig)
        return list(val) if isinstance(val, (list, tuple)) else val
    except Exception:
        pass

    # Make a cleaned copy: normalize smart quotes, remove non-printable control chars
    s2 = orig.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    s2 = re.sub(r"[\x00-\x1f\x7f-\x9f]", "", s2)  # strip control chars

    # 2) Remove trailing commas before closing bracket: [a, b,] -> [a, b]
    s2 = re.sub(r",\s*(\])", r"\1", s2)

    # 3) If it starts with [ but missing a closing ], append it
    if s2.strip().startswith("[") and not s2.strip().endswith("]"):
        s2 = s2 + "]"

    # 4) Try json.loads after converting single quotes to double quotes (common fix)
    try:
        candidate = s2.replace("'", '"')
        # also collapse duplicated commas like ", ,"
        candidate = re.sub(r",\s*,+", ",", candidate)
        val = json.loads(candidate)
        return list(val) if isinstance(val, (list, tuple)) else val
    except Exception:
        pass

    # 5) Try ast.literal_eval again on repaired s2
    try:
        val = ast.literal_eval(s2)
        return list(val) if isinstance(val, (list, tuple)) else val
    except Exception:
        pass

    # 6) If quotes are unbalanced, try to balance single quotes by appending a quote/bracket
    s3 = s2
    if s3.count("'") % 2 == 1:
        # append a closing single-quote and bracket if needed
        if not s3.strip().endswith("'"):
            s3 = s3 + "'"
        if s3.strip().startswith("[") and not s3.strip().endswith("]"):
            s3 = s3 + "]"
    try:
        val = ast.literal_eval(s3)
        return list(val) if isinstance(val, (list, tuple)) else val
    except Exception:
        pass

    # 7) FINAL TOLERANT FALLBACK: extract comma-separated tokens, honoring quoted substrings
    # This will capture 'name', "name", or unquoted tokens between commas/brackets
    tokens = []
    for m in re.finditer(r"""
            '([^']*)'       |   # single-quoted token
            "([^"]*)"       |   # double-quoted token
            ([^\[\],]+)         # unquoted token (stop at comma or bracket)
        """, orig, re.VERBOSE):
        # pick the non-empty capture
        token = m.group(1) or m.group(2) or m.group(3)
        token = token.strip()
        if token:
            tokens.append(token)
    # If still empty, return empty list rather than None
    return tokens

# Apply to dataframe and record which indices needed repair
problematic_indices = []
repair_method = {}

for idx, val in tqdm(df['researchers'].items(), total=len(df)):
    parsed = safe_parse_researchers(val)
    # ensure it's a list (if single string, wrap it)
    if parsed is None:
        parsed = []
    if isinstance(parsed, (str, int, float)):
        parsed = [parsed]
    df.at[idx, 'researchers'] = parsed

# Optionally, find rows that are still empty or suspicious after parsing
still_bad = df[df['researchers'].apply(lambda x: (not isinstance(x, list)) or any(pd.isna(i) for i in x) )].index.tolist()
print(f"Total rows parsed: {len(df)}")
print(f"Rows still suspicious after repair: {len(still_bad)} (examples: {still_bad[:20]})")

df['researcher_count'] = df['researchers'].progress_apply(lambda x: len(x) if str(x)!= '[]' else 0)

100%|██████████| 10846/10846 [00:09<00:00, 1188.97it/s]


Total rows parsed: 10846
Rows still suspicious after repair: 0 (examples: [])


  0%|          | 0/10846 [00:00<?, ?it/s]

In [4]:

# Apply to dataframe and record which indices needed repair
problematic_indices = []
repair_method = {}

for idx, val in tqdm(df['research_orgs'].items(), total=len(df)):
    parsed = safe_parse_researchers(val)
    # ensure it's a list (if single string, wrap it)
    if parsed is None:
        parsed = []
    if isinstance(parsed, (str, int, float)):
        parsed = [parsed]
    df.at[idx, 'research_orgs'] = parsed

# Optionally, find rows that are still empty or suspicious after parsing
still_bad = df[df['research_orgs'].apply(lambda x: (not isinstance(x, list)) or any(pd.isna(i) for i in x) )].index.tolist()
print(f"Total rows parsed: {len(df)}")
print(f"Rows still suspicious after repair: {len(still_bad)} (examples: {still_bad[:20]})")

100%|██████████| 10846/10846 [00:03<00:00, 2752.48it/s]


Total rows parsed: 10846
Rows still suspicious after repair: 0 (examples: [])


In [5]:
# Examine the data structure
print("Sample researcher entry:")
print(df['researchers'][0][0])
print("\nData type:", type(df['researchers'][0]))
print("Number of researchers in first paper:", len(df['researchers'][0]) if isinstance(df['researchers'][0], list) else 0)

Sample researcher entry:
{'first_name': 'Xian-Wen', 'id': 'ur.012214661411.72', 'last_name': 'Shang', 'orcid_id': ['0000-0001-6143-9349', '0000-0002-2362-3222'], 'research_orgs': ['grid.43169.39', 'grid.12981.33', 'grid.415105.4', 'grid.410670.4', 'grid.284723.8', 'grid.1008.9', 'grid.416153.4', 'grid.16890.36', 'grid.508448.5', 'grid.411958.0', 'grid.1103439.3d', 'grid.506261.6', 'grid.198530.6', 'grid.418002.f', 'grid.413352.2']}

Data type: <class 'list'>
Number of researchers in first paper: 12


# Research Plan: UK Biobank Network Analysis Over Time

## Data Structure Update
The `researchers` column contains a **list of dictionaries** with the following structure:
- `id`: Unique researcher identifier (e.g., 'ur.012214661411.72')
- `first_name`: Researcher's first name
- `last_name`: Researcher's last name
- `orcid_id`: List of ORCID identifiers
- `research_orgs`: List of organization IDs associated with this researcher

Similarly, `research_orgs` likely contains organization information.

## Network Analysis Approach
We'll build **co-authorship networks** where:
- **Nodes** = researchers (identified by their `id`)
- **Edges** = collaboration (researchers who co-authored papers)

## Key Metrics to Track Over Time:
- Total unique authors
- Total collaborations (edges)
- Giant component size (largest connected group)
- Number of isolates (single-author papers)
- Number of connected components
- Network density & clustering

In [6]:
# Install/import required packages
import networkx as nx
from scipy import sparse as sp
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns
from collections import Counter

print("NetworkX version:", nx.__version__)
print("Number of papers:", len(df))
print("Year range:", df['year'].min(), "to", df['year'].max())

NetworkX version: 3.6
Number of papers: 10846
Year range: 2013 to 2025


In [13]:
def build_author_network(df_subset):
    """
    Build author-author co-authorship network from papers.
    Each researcher is identified by their 'id' field in the researcher dictionary.
    
    Based on PopStudies approach: create bipartite author-paper matrix,
    then project to author-author network where edges represent co-authorship.
    """
    # Filter papers with valid researchers
    df_valid = df_subset[df_subset['researchers'].apply(
        lambda x: isinstance(x, list) and len(x) > 0)].copy()
    
    if len(df_valid) == 0:
        return None, {}
    
    # Extract unique papers and researchers
    unique_papers = df_valid['id'].unique().tolist()
    unique_authors = set()
    
    # Build author info dictionary: author_id -> researcher dict
    author_info = {}
    
    for researchers_list in df_valid['researchers']:
        if isinstance(researchers_list, list):
            for researcher in researchers_list:
                if isinstance(researcher, dict) and 'id' in researcher:
                    author_id = researcher['id']
                    unique_authors.add(author_id)
                    if author_id not in author_info:
                        author_info[author_id] = researcher
    
    unique_authors = sorted(unique_authors)
    
    if len(unique_authors) == 0:
        return None, {}
    
    # Create integer mappings for matrix
    paper_to_int = {paper: idx for idx, paper in enumerate(unique_papers)}
    author_to_int = {author: idx for idx, author in enumerate(unique_authors)}
    int_to_author = {idx: author for author, idx in author_to_int.items()}
    
    # Build author-paper tuples
    author_paper_tuples = []
    for _, row in df_valid.iterrows():
        paper_id = paper_to_int[row['id']]
        researchers_list = row['researchers']
        
        if isinstance(researchers_list, list):
            for researcher in researchers_list:
                if isinstance(researcher, dict) and 'id' in researcher:
                    author_id = researcher['id']
                    author_idx = author_to_int[author_id]
                    author_paper_tuples.append((author_idx, paper_id))
    
    if len(author_paper_tuples) == 0:
        return None, {}
    
    # Create sparse bipartite author-paper matrix
    n_authors = len(unique_authors)
    n_papers = len(unique_papers)
    
    author_ids, paper_ids = zip(*author_paper_tuples)
    AP = sp.csc_matrix(
        (np.ones(len(author_paper_tuples)), (author_ids, paper_ids)),
        shape=(n_authors, n_papers)
    )
    
    # Project to author-author network: AA = AP * AP^T
    # This creates edges between authors who co-authored papers
    AA = AP.dot(AP.T)
    
    # Remove self-loops (diagonal elements)
    AA = AA - sp.diags(AA.diagonal())
    
    # Convert to NetworkX graph
    G = nx.from_scipy_sparse_array(AA)
    
    # Return both graph and mapping with author info
    author_mapping = {
        'int_to_author': int_to_author,
        'author_to_int': author_to_int,
        'author_info': author_info
    }
    
    return G, author_mapping


In [14]:
def compute_network_metrics(G):
    """
    Compute comprehensive network metrics including tie strength and structural properties.
    Returns a dictionary with extensive network statistics.
    """
    if G is None or G.number_of_nodes() == 0:
        return {
            'n_nodes': 0,
            'n_edges': 0,
            'density': 0,
            'n_components': 0,
            'n_isolates': 0,
            'giant_nodes': 0,
            'giant_edges': 0,
            'giant_density': 0,
            'avg_clustering': 0,
            'avg_degree': 0,
            'max_degree': 0,
            'avg_tie_strength': 0,
            'max_tie_strength': 0,
            'median_tie_strength': 0,
            'transitivity': 0,
            'assortativity': 0,
            'diameter': 0,
            'avg_shortest_path': 0,
            'n_weak_ties': 0,
            'n_strong_ties': 0,
            'weak_tie_ratio': 0
        }
    
    # ============ Basic metrics ============
    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()
    density = nx.density(G)
    
    # ============ Degree distribution ============
    degrees = [d for n, d in G.degree()]
    avg_degree = np.mean(degrees) if len(degrees) > 0 else 0
    max_degree = max(degrees) if len(degrees) > 0 else 0
    
    # ============ Connected components ============
    components = list(nx.connected_components(G))
    n_components = len(components)
    
    # Isolates (nodes with degree 0)
    isolates = list(nx.isolates(G))
    n_isolates = len(isolates)
    
    # Giant component (largest connected component)
    if n_components > 0:
        giant_component = max(components, key=len)
        G_giant = G.subgraph(giant_component)
        giant_nodes = G_giant.number_of_nodes()
        giant_edges = G_giant.number_of_edges()
        giant_density = nx.density(G_giant) if giant_nodes > 1 else 0
    else:
        giant_nodes = giant_edges = giant_density = 0
        G_giant = None
    
    # ============ Clustering ============
    """ 
    try:
        avg_clustering = nx.average_clustering(G)
        transitivity = nx.transitivity(G)  # Global clustering coefficient
    except:
        avg_clustering = 0
        transitivity = 0
    """
    # ============ Tie Strength Analysis ============
    # Tie strength = edge weight (number of co-authored papers)
    tie_strengths = []
    for u, v, data in G.edges(data=True):
        weight = data.get('weight', 1)  # Default weight is 1
        tie_strengths.append(weight)
    
    if len(tie_strengths) > 0:
        avg_tie_strength = np.mean(tie_strengths)
        max_tie_strength = max(tie_strengths)
        median_tie_strength = np.median(tie_strengths)
        
        # Define weak vs strong ties (threshold: median or mean)
        threshold = median_tie_strength
        n_weak_ties = sum(1 for w in tie_strengths if w <= threshold)
        n_strong_ties = sum(1 for w in tie_strengths if w > threshold)
        weak_tie_ratio = n_weak_ties / len(tie_strengths) if len(tie_strengths) > 0 else 0
    else:
        avg_tie_strength = max_tie_strength = median_tie_strength = 0
        n_weak_ties = n_strong_ties = weak_tie_ratio = 0
    
    # ============ Assortativity (degree correlation) ============
    """ 
    try:
        assortativity = nx.degree_assortativity_coefficient(G)
    except:
        assortativity = 0
    """
    # ============ Path-based metrics (only for giant component if network is disconnected) ============
    # COMMENTED OUT: These are computationally expensive O(n²) operations
    """
    diameter = 0
    avg_shortest_path = 0
    
    if G_giant is not None and giant_nodes > 1:
        try:
            # Diameter (longest shortest path in giant component)
            diameter = nx.diameter(G_giant)
        except:
            diameter = 0
        
        try:
            # Average shortest path length (giant component)
            avg_shortest_path = nx.average_shortest_path_length(G_giant)
        except:
            avg_shortest_path = 0
    """
    # Set to 0 when commented out
    diameter = 0
    avg_shortest_path = 0
    
    return {
        # Basic structure
        'n_nodes': n_nodes,
        'n_edges': n_edges,
        'density': density,
        
        # Components
        'n_components': n_components,
        'n_isolates': n_isolates,
        'giant_nodes': giant_nodes,
        'giant_edges': giant_edges,
        'giant_density': giant_density,
        
        # Degree metrics
        'avg_degree': avg_degree,
        'max_degree': max_degree,
        
        # Clustering
        #'avg_clustering': avg_clustering,
        #'transitivity': transitivity,
        
        # Tie strength
        'avg_tie_strength': avg_tie_strength,
        'max_tie_strength': max_tie_strength,
        'median_tie_strength': median_tie_strength,
        'n_weak_ties': n_weak_ties,
        'n_strong_ties': n_strong_ties,
        'weak_tie_ratio': weak_tie_ratio,
        
        # Network structure
        #'assortativity': assortativity,
        'diameter': diameter,
        'avg_shortest_path': avg_shortest_path
    }


In [15]:
# Build temporal networks: cumulative year-by-year analysis
# This tracks how the network grows over time

# Ensure year is numeric and sort
df_clean = df[df['year'].notna()].copy()
df_clean['year'] = df_clean['year'].astype(int)
df_clean = df_clean.sort_values('year')

# Get year range
min_year = int(df_clean['year'].min())
max_year = int(df_clean['year'].max())

print(f"Analyzing network evolution from {min_year} to {max_year}")
print(f"Total papers: {len(df_clean)}")
print(f"Papers with researchers: {len(df_clean[df_clean['researcher_count'] > 0])}")

# Initialize results storage
years = range(min_year, max_year + 1)
metrics_over_time = []


# Build cumulative networks year by year
print("\nBuilding temporal networks...")
for year in tqdm(years):
    # Cumulative: all papers up to and including this year
    df_upto_year = df_clean[df_clean['year'] <= year]
    
    # Build network
    G, author_mapping = build_author_network(df_upto_year)
    
    # Compute metrics
    metrics = compute_network_metrics(G)
    metrics['year'] = year
    metrics['n_papers'] = len(df_upto_year)
    
    metrics_over_time.append(metrics)

# Convert to dataframe
network_metrics_df = pd.DataFrame(metrics_over_time)
network_metrics_df = network_metrics_df.set_index('year')

print("\n" + "="*60)
print("Network metrics computed successfully!")
print("="*60)
print(network_metrics_df.head(10))

Analyzing network evolution from 2013 to 2025
Total papers: 10846
Papers with researchers: 10801

Building temporal networks...


100%|██████████| 13/13 [00:18<00:00,  1.46s/it]


Network metrics computed successfully!
      n_nodes  n_edges   density  n_components  n_isolates  giant_nodes  \
year                                                                      
2013       31      132  0.283871             4           0           14   
2014       64      280  0.138889             7           0           18   
2015      172     1136  0.077247            16           0           35   
2016      792    11447  0.036544            14           0          644   
2017     1685    24657  0.017379            29           1         1526   
2018     3244    51654  0.009820            37           3         2986   
2019     5224    82974  0.006082            60           4         4808   
2020     8683   141031  0.003742           111           4         7897   
2021    13069   211437  0.002476           155           6        12000   
2022    18024   293095  0.001805           201           9        16688   

      giant_edges  giant_density  avg_degree  max_degree  a

## Part 2: Cumulative Network Evolution & Animation

Now we'll create visualizations showing how the **cumulative network grows over time** - each year includes all papers published up to that year. We'll generate an animated GIF showing network evolution.

## Enhanced GIF Animation with Metrics Panel

Create an animated GIF showing:
- **Left**: Network visualization (cumulative, showing giant component & isolates)
- **Right**: 3 metric plots vertically stacked:
  1. Network nodes and edges
  2. Average collaborators per author
  3. Giant component growth

Each frame shows a **sliding time window** (e.g., past X years) to keep the right panel focused and readable.

In [16]:
# Configuration for enhanced GIF with metrics panel
import os
from PIL import Image

# Time window for x-axis (years to show in metrics plots)
TIME_WINDOW_YEARS = 12  # Show past 12 years in each frame

# Create directory for enhanced frames
enhanced_frames_dir = '../data/network_frames_with_metrics'
os.makedirs(enhanced_frames_dir, exist_ok=True)
min_year = 2015 
max_year = 2025
print(f"Configuration:")
print(f"  Time window: {TIME_WINDOW_YEARS} years")
print(f"  Frames directory: {enhanced_frames_dir}")

plt.rcParams['font.family'] = 'Helvetica'

Configuration:
  Time window: 12 years
  Frames directory: ../data/network_frames_with_metrics


In [29]:
# Generate enhanced frames: Network (left) + Metrics (right)

print("Generating enhanced network snapshots with metrics panels...")
print("This may take several minutes for large networks...")

# Select years to visualize (every year)
years_to_plot = list(range(min_year, max_year + 1, 1))
if max_year not in years_to_plot:
    years_to_plot.append(max_year)

enhanced_frame_files = []

for year in tqdm(years_to_plot):
    # Build CUMULATIVE network up to this year
    df_upto_year = df_clean[df_clean['year'] <= year]
    G, author_mapping = build_author_network(df_upto_year)
    
    if G is None or G.number_of_nodes() == 0:
        continue
    
    # Create figure with GridSpec: left for network, right for metrics
    fig = plt.figure(figsize=(16,9))
    gs = fig.add_gridspec(4, 2, width_ratios=[2, 1], hspace=0.35, wspace=0.3,
                          left=0.05, right=0.97, top=0.93, bottom=0.07)
    
    # ========== LEFT: Network Visualization ==========
    ax_network = fig.add_subplot(gs[:, 0])
    
    # Get components for coloring
    components = list(nx.connected_components(G))
    components_sorted = sorted(components, key=len, reverse=True)
    
    # Color scheme
    
    colors_scheme = [ "#B80C09", "#D4AF37", "#6E8B3D","#345995"]
        
    # Assign colors and sizes based on component type
    node_colors = []
    node_sizes = []
    
    for node in G.nodes():
        component_idx = None
        for idx, comp in enumerate(components_sorted):
            if node in comp:
                component_idx = idx
                break
        
        comp_size = len(components_sorted[component_idx]) if component_idx is not None else 1
        
        if comp_size == 1:  # Isolates
            node_colors.append(colors_scheme[0])
            node_sizes.append(20)
        elif comp_size <= 5:
            node_colors.append(colors_scheme[1])
            node_sizes.append(30)
        elif component_idx == 0:  # Giant component
            node_colors.append(colors_scheme[3])
            node_sizes.append(35)
        else:
            node_colors.append(colors_scheme[2])
            node_sizes.append(25)
    
    # Compute layout
    iterations = 20 if G.number_of_nodes() > 1000 else 40
    pos = nx.spring_layout(G, k=0.3, iterations=iterations, seed=48652)
    
    # Draw network
    nx.draw_networkx_nodes(G, pos, 
                          node_color=node_colors, 
                          node_size=node_sizes,
                          alpha=0.7,
                          edgecolors='k',
                          linewidths=0.5,
                          ax=ax_network)
    
    nx.draw_networkx_edges(G, pos, 
                          alpha=0.15, 
                          width=0.3,
                          ax=ax_network)
    
    # Network title - positioned at bottom
    metrics_current = network_metrics_df.loc[year]
    title_text = (f'UK Biobank Co-authorship Network (Through {year})\n' + 
                  f'{metrics_current["n_nodes"]:,.0f} authors • ' +
                  f'{metrics_current["n_edges"]:,.0f} collaborations • ' +
                  f'Giant Component: {metrics_current["giant_nodes"]:,.0f} ' +
                  f'({100*metrics_current["giant_nodes"]/metrics_current["n_nodes"]:.1f}%)')
    
    ax_network.text(0.5, 0.01, title_text, transform=ax_network.transAxes,
                    fontsize=13, fontweight='bold', ha='center', va='bottom')
                    #bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='k', lw=1))
    
    ax_network.axis('off')
    
    # Add legend
    legend_elements = [
        mlines.Line2D([], [], color=colors_scheme[0], marker='o', linestyle='None',
                     markersize=10, label='Isolates', markeredgecolor='k', markeredgewidth=0.5, alpha=0.7),
        # mark on other colors too 
        mlines.Line2D([], [], color=colors_scheme[1], marker='o', linestyle='None',
                     markersize=10, label='2-5 Small Collaborations', markeredgecolor='k', markeredgewidth=0.5, alpha=0.7),
        mlines.Line2D([], [], color=colors_scheme[2], marker='o', linestyle='None',
                     markersize=10, label='Other Medium Collaborations', markeredgecolor='k', markeredgewidth=0.5, alpha=0.7),
        mlines.Line2D([], [], color=colors_scheme[3], marker='o', linestyle='None',
                     markersize=10, label='Giant Component', markeredgecolor='k', markeredgewidth=0.5, alpha=0.7)
    ]
    ax_network.legend(handles=legend_elements,bbox_to_anchor=(0.85, 0.1), frameon=True, 
                     framealpha=0.95, edgecolor='k', fontsize=11)
    
    # ========== RIGHT: Metrics Panels (3 stacked vertically) ==========
    
    # Determine time window for x-axis
    x_min = max(min_year, year - TIME_WINDOW_YEARS + 1)
    x_max = max_year  # Always show full x-axis to max_year
    
    # Filter data for this time window (up to current year)
    data_mask = (network_metrics_df.index >= x_min) & (network_metrics_df.index <= year)
    data_window = network_metrics_df[data_mask]
    

    # Panel 0: Nodes 
    ax0 = fig.add_subplot(gs[0, 1])
    
    ax0.plot(data_window.index, data_window['n_nodes'], 
             color=colors_scheme[2], linewidth=2.5, marker='o', markersize=4, label='Authors')
    ax0.fill_between(data_window.index, 0, data_window['n_nodes'], alpha=0.3, color=colors_scheme[2])
    
    #ax0.set_ylabel('Authors', fontsize=13, fontweight='bold')
    ax0.tick_params(labelsize=9)

    ax0.set_ylim(-5,40000)
    ax0.set_xlim(x_min-0.5, x_max+1)
    ax0.set_title('Network Growth - Authors', fontsize=12, fontweight='bold', loc='left', pad=8)
    ax0.grid(alpha=0.3, linestyle='--', linewidth=0.5)
    ax0.tick_params(axis='x', labelsize=9)
    ax0.set_xlabel('')
    
    
    # Panel 1: Nodes and Edges
    ax1 = fig.add_subplot(gs[1, 1])
    
    ax1.plot(data_window.index, data_window['n_edges'], 
                  color="#B80C09", linewidth=2.5, marker='o', markersize=4, label='Collaborations')
    ax1.fill_between(data_window.index, 0, data_window['n_edges'], 
                     alpha=0.3, color="#B80C09")
    #ax1.set_ylabel('Collaborations', fontsize=13, fontweight='bold')
    ax1.set_ylim(-5,580000)
    ax1.tick_params(labelsize=9)
    ax1.set_xlim(x_min-0.5, x_max+1)
    ax1.set_title('Network Growth - Collaborations', fontsize=12, fontweight='bold', loc='left', pad=8)
    ax1.grid(alpha=0.3, linestyle='--', linewidth=0.5)
    ax1.tick_params(axis='x', labelsize=9)
    ax1.set_xlabel('')
    
    # Panel 2: Average Degree (Collaborators per Author)
    ax2 = fig.add_subplot(gs[2, 1])
    
    ax2.plot(data_window.index, data_window['avg_degree'], 
             color= "#D4AF37", linewidth=2.5, marker='o', markersize=4)
    ax2.fill_between(data_window.index, 0, data_window['avg_degree'],
                     alpha=0.3, color= "#D4AF37")
    
    #ax2.set_ylabel('Avg Degree', fontsize=13, fontweight='bold')
    ax2.set_xlim(x_min-0.5,x_max+1)
    ax2.set_title('Avg Collaborators per Author', fontsize=12, fontweight='bold', loc='left', pad=8)
    ax2.grid(alpha=0.3, linestyle='--', linewidth=0.5)
    ax2.tick_params(labelsize=9)
    ax2.set_ylim(0,33)
    ax2.set_xlabel('')
    
    # Panel 3: Giant Component Growth
    ax3 = fig.add_subplot(gs[3, 1])
    
    ax3.plot(data_window.index, data_window['giant_nodes'], 
             color=colors_scheme[3], linewidth=2.5, marker='o', markersize=4)
    ax3.fill_between(data_window.index, 0, data_window['giant_nodes'],
                     alpha=0.3, color=colors_scheme[3])
    
    #ax3.set_ylabel('Giant Component', fontsize=13, fontweight='bold')
    ax3.set_xlabel('Year', fontsize=13, fontweight='bold')
    ax3.set_xlim(x_min-0.5, x_max+1)
    ax3.set_title('Giant Component Size', fontsize=12, fontweight='bold', loc='left', pad=8)
    ax3.grid(alpha=0.3, linestyle='--', linewidth=0.5)
    ax3.tick_params(labelsize=9)
    ax3.set_ylim(-5,35000)
    
    # Apply styling
    sns.despine(ax=ax1)
    sns.despine(ax=ax0)
    sns.despine(ax=ax2)
    sns.despine(ax=ax3)

    
    # Save frame
    frame_file = os.path.join(enhanced_frames_dir, f'network_enhanced_{year}.png')
    if year in [min_year, max_year]:
        print(f" Saving frame for year {year} to {frame_file}")
        plt.savefig(os.path.join(enhanced_frames_dir, f'network_enhanced_{year}.svg'), bbox_inches='tight', dpi=300, facecolor='white')
        
    plt.savefig(frame_file, bbox_inches='tight', dpi=300, facecolor='white')
    plt.close()
    
    enhanced_frame_files.append(frame_file)
  
 

Generating enhanced network snapshots with metrics panels...
This may take several minutes for large networks...


  0%|          | 0/11 [00:00<?, ?it/s]

 Saving frame for year 2015 to ../data/network_frames_with_metrics/network_enhanced_2015.png


 91%|█████████ | 10/11 [58:27<13:34, 814.38s/it]

 Saving frame for year 2025 to ../data/network_frames_with_metrics/network_enhanced_2025.png


100%|██████████| 11/11 [1:37:51<00:00, 533.73s/it] 


In [ ]:
components = list(nx.connected_components(G))
components_sorted = sorted(components, key=len, reverse=True)

In [23]:
for i in range(len(components_sorted)):
    print(f"Component {i+1}: Size {len(components_sorted[i])}")

Component 1: Size 35
Component 2: Size 28
Component 3: Size 25
Component 4: Size 16
Component 5: Size 14
Component 6: Size 11
Component 7: Size 10
Component 8: Size 6
Component 9: Size 5
Component 10: Size 5
Component 11: Size 4
Component 12: Size 4
Component 13: Size 3
Component 14: Size 2
Component 15: Size 2
Component 16: Size 2


In [20]:
len(components_sorted)

16

In [30]:
# Create enhanced animated GIF

print("Creating enhanced animated GIF with metrics panel...")

# Load all enhanced frames
enhanced_frames = []
for frame_file in enhanced_frame_files:
    img = Image.open(frame_file)
    enhanced_frames.append(img)

# Save as GIF
enhanced_gif_path = '../fig/network/network_evolution_with_metrics.gif'
enhanced_frames[0].save(
    enhanced_gif_path,
    save_all=True,
    append_images=enhanced_frames[1:],
    duration=2000,  # milliseconds per frame 
    loop=0,
    optimize=False
)


Creating enhanced animated GIF with metrics panel...


## Leiden community-aggregated co-authorship network, 2025

This section reuses the parsed `df_clean` data and the existing `build_author_network` function above. It builds one 2025 author-level graph, detects Leiden communities on the full graph, stores author membership, collapses the result to the largest communities plus an `Other Communities` node, and saves publication-ready outputs.


In [ ]:
# Leiden community network configuration for the 2025 co-authorship graph
from pathlib import Path
from collections import Counter
import ast
import math
import textwrap
import warnings

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patheffects as pe

try:
    import igraph as ig
    import leidenalg
except ImportError as exc:
    raise ImportError(
        "This analysis requires python-igraph and leidenalg. "
        "Install them in the ukbb20 environment with: "
        "conda install -n ukbb20 -c conda-forge python-igraph leidenalg"
    ) from exc

# The existing notebook builds cumulative networks "through year". Keeping this as
# True reproduces the final 2025 network used in the temporal analysis. Set to
# False only if you want papers published in 2025 alone.
ANALYSIS_YEAR = 2025
USE_CUMULATIVE_NETWORK_THROUGH_YEAR = True

# Retain roughly 10 large communities as individual nodes; merge all remaining
# communities into one aggregate node for readability.
N_LARGEST_COMMUNITIES_TO_KEEP = 10
OTHER_COMMUNITIES_LABEL = "Other Communities"

# Leave as None to merge every non-top community into Other Communities. Set to
# 20 if you prefer to omit communities smaller than 20 authors from the plot.
OMIT_COMMUNITIES_SMALLER_THAN = None

LEIDEN_RESOLUTION = 1.0
RANDOM_SEED = 48652
LABEL_TOP_N_COMMUNITIES = 10
LABEL_MIN_AUTHORS = 500
MIXED_TOPIC_LABEL_SHARE_THRESHOLD = 0.55
TOPIC_LABEL_OVERRIDES = {
    "Biomedical and Clinical Sciences": "Biomedical/Clinical",
    "Biological Sciences": "Biological",
    "Health Sciences": "Health",
    "Information and Computing Sciences": "Computing",
    "Chemical Sciences": "Chemistry",
    "Environmental Sciences": "Environmental",
    "Cardiovascular Medicine and Haematology": "Cardiovascular/Haematology",
    "Health Services and Systems": "Health Services",
    "Ophthalmology and Optometry": "Ophthalmology/Optometry",
    "Pharmacology and Pharmaceutical Sciences": "Pharmacology",
    "Medical Biochemistry and Metabolomics": "Medical Biochemistry",
    "Bioinformatics and Computational Biology": "Bioinformatics",
    "Clinical and Health Psychology": "Clinical Psychology",
    "Cognitive and Computational Psychology": "Cognitive Psychology",
}
INSTITUTION_LABEL_OVERRIDES = {
    "University of Oxford": "Oxford",
    "University of Cambridge": "Cambridge",
    "University College London": "UCL",
    "Imperial College London": "Imperial",
    "King's College London": "King's College London",
    "Queen Mary University of London": "Queen Mary London",
    "University of Manchester": "Manchester",
    "University of Edinburgh": "Edinburgh",
    "University of Bristol": "Bristol",
    "University of Glasgow": "Glasgow",
    "University of Exeter": "Exeter",
    "University of Liverpool": "Liverpool",
}

COMMUNITY_OUTPUT_DIR = Path("../data/network_communities_2025")
FIG_OUTPUT_DIR = Path("../fig/network")
COMMUNITY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.edgecolor": "#222222",
    "axes.linewidth": 0.8,
})


In [ ]:
def build_author_graph_for_analysis_year(df_clean, analysis_year=2025, cumulative=True):
    """
    Build the author-level NetworkX graph for one analysis year using the
    notebook's existing build_author_network() pipeline.
    """
    if cumulative:
        df_year = df_clean[df_clean["year"] <= analysis_year].copy()
        slice_label = f"Through {analysis_year}"
    else:
        df_year = df_clean[df_clean["year"] == analysis_year].copy()
        slice_label = f"Published in {analysis_year}"

    graph, author_mapping = build_author_network(df_year)
    if graph is None or graph.number_of_nodes() == 0:
        raise ValueError(f"No author graph could be built for {slice_label}.")

    # Ensure edge weights are plain floats for igraph, plotting, and CSV export.
    for _, _, data in graph.edges(data=True):
        data["weight"] = float(data.get("weight", 1.0))

    return graph, author_mapping, df_year, slice_label


def networkx_to_igraph(graph, weight_attr="weight"):
    """
    Convert an undirected weighted NetworkX graph to igraph while preserving a
    mapping back to the original NetworkX node IDs.
    """
    nodes = list(graph.nodes())
    node_to_idx = {node: idx for idx, node in enumerate(nodes)}
    edges = [(node_to_idx[u], node_to_idx[v]) for u, v in graph.edges()]
    weights = [float(data.get(weight_attr, 1.0)) for _, _, data in graph.edges(data=True)]

    ig_graph = ig.Graph(n=len(nodes), edges=edges, directed=False)
    ig_graph.vs["nx_node"] = nodes
    ig_graph.vs["name"] = [str(node) for node in nodes]
    if weights:
        ig_graph.es["weight"] = weights

    return ig_graph, nodes


def detect_leiden_communities(graph, resolution=1.0, seed=48652, weight_attr="weight"):
    """
    Run Leiden community detection on the full author-level graph.

    Returns author-node membership, the Leiden partition, the igraph graph, and
    weighted modularity for the detected partition.
    """
    ig_graph, nx_nodes = networkx_to_igraph(graph, weight_attr=weight_attr)
    weights = "weight" if ig_graph.ecount() > 0 else None

    partition = leidenalg.find_partition(
        ig_graph,
        leidenalg.RBConfigurationVertexPartition,
        weights=weights,
        resolution_parameter=resolution,
        n_iterations=-1,
        seed=seed,
    )

    membership_by_node = dict(zip(nx_nodes, partition.membership))
    nx.set_node_attributes(graph, membership_by_node, "leiden_community")

    modularity_weights = ig_graph.es["weight"] if ig_graph.ecount() > 0 else None
    modularity = ig_graph.modularity(partition.membership, weights=modularity_weights)

    return membership_by_node, partition, ig_graph, modularity


def researcher_display_name(author_id, author_info):
    """Return a readable author name when the Dimensions record contains one."""
    info = author_info.get(author_id, {})
    if not isinstance(info, dict):
        return pd.NA

    for key in ("full_name", "name", "display_name"):
        if info.get(key):
            return info[key]

    first = info.get("first_name") or info.get("firstName")
    last = info.get("last_name") or info.get("lastName")
    if first or last:
        return " ".join(part for part in [first, last] if part)

    return pd.NA


def make_author_community_table(membership_by_node, author_mapping):
    """Create one row per author with original author ID and Leiden community."""
    int_to_author = author_mapping.get("int_to_author", {})
    author_info = author_mapping.get("author_info", {})

    records = []
    for node, community_id in membership_by_node.items():
        author_id = int_to_author.get(node, node)
        records.append({
            "author_node": node,
            "author_id": author_id,
            "author_name": researcher_display_name(author_id, author_info),
            "leiden_community": int(community_id),
        })

    return pd.DataFrame(records).sort_values(
        ["leiden_community", "author_id"], kind="mergesort"
    ).reset_index(drop=True)


def parse_listcol_for_topics(value):
    """Parse Dimensions list-like fields used for topic extraction."""
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []


def split_for_name(name):
    """Return (code, label, level) from a FOR 2020 category name."""
    if not isinstance(name, str) or not name.strip():
        return None, None, None
    parts = name.strip().split(" ", 1)
    code = parts[0]
    label = parts[1].strip() if len(parts) > 1 else ""
    if code.isdigit() and len(code) == 2:
        return code, label, "L2"
    if code.isdigit() and len(code) == 4:
        return code, label, "L4"
    return None, None, None


def build_paper_topic_table(paper_df):
    """
    Derive each paper's primary FOR L2 and L4 labels from category_for_2020.

    L2 follows 01_analysis_03_authors.ipynb: the first-listed category's L2
    division, resolving L4 prefixes back to their division. L4 is the first
    usable L4 category on the paper, which gives more distinctive labels for
    communities than broad L2 divisions.
    """
    if "category_for_2020" not in paper_df.columns:
        raise KeyError("category_for_2020 is needed to derive primary_for_l2 topics.")

    parsed_categories = paper_df["category_for_2020"].apply(parse_listcol_for_topics)
    div_lut = {}
    for categories in parsed_categories:
        for category in categories:
            if not isinstance(category, dict):
                continue
            code, label, level = split_for_name(category.get("name"))
            if level == "L2" and code not in div_lut:
                div_lut[code] = label

    rows = []
    for paper_id, categories in zip(paper_df["id"], parsed_categories):
        l2_topics, l4_topics = [], []
        primary_l2, primary_l4 = None, None
        for category in categories:
            if not isinstance(category, dict):
                continue
            code, label, level = split_for_name(category.get("name"))
            if level == "L2":
                l2_topics.append(label)
                if primary_l2 is None:
                    primary_l2 = label
            elif level == "L4":
                l4_topics.append(label)
                if primary_l2 is None:
                    primary_l2 = div_lut.get(code[:2])
                if primary_l4 is None:
                    primary_l4 = label
        rows.append({
            "paper_id": paper_id,
            "primary_for_l2": primary_l2 or "Unclassified",
            "primary_for_l4": primary_l4 or "Unclassified",
            "for_l2": sorted(set(l2_topics)),
            "for_l4": sorted(set(l4_topics)),
        })

    return pd.DataFrame(rows)


def get_paper_topic_source(df_all=None, path="../data/df_dimensions.xlsx"):
    """Return paper IDs and category_for_2020, loading them if needed."""
    if df_all is not None and {"id", "category_for_2020"}.issubset(df_all.columns):
        return df_all[["id", "category_for_2020"]].copy()
    return pd.read_excel(path, usecols=["id", "category_for_2020"])


def make_author_modal_topic_table(df_year, paper_topics_df):
    """Assign each author modal primary_for_l2 and primary_for_l4 topics."""
    topic_cols = ["primary_for_l2", "primary_for_l4"]
    paper_topics = paper_topics_df[["paper_id"] + topic_cols].copy()
    for col in topic_cols:
        paper_topics[col] = paper_topics[col].fillna("Unclassified")
    paper_topic_lookup = paper_topics.set_index("paper_id")[topic_cols].to_dict("index")

    records = []
    for _, row in df_year.iterrows():
        paper_id = row["id"]
        topics = paper_topic_lookup.get(
            paper_id,
            {"primary_for_l2": "Unclassified", "primary_for_l4": "Unclassified"},
        )
        for researcher in parse_listcol_for_topics(row["researchers"]):
            if not isinstance(researcher, dict):
                continue
            author_id = researcher.get("id") or researcher.get("researcher_id")
            if author_id:
                records.append({
                    "author_id": author_id,
                    "paper_id": paper_id,
                    "primary_for_l2": topics.get("primary_for_l2", "Unclassified"),
                    "primary_for_l4": topics.get("primary_for_l4", "Unclassified"),
                })

    author_papers = pd.DataFrame(records).drop_duplicates(["author_id", "paper_id"])
    if author_papers.empty:
        return pd.DataFrame(columns=["author_id"])

    def modal_for(topic_col):
        count_col = f"modal_{topic_col}_papers"
        total_col = f"{topic_col}_observed_papers"
        n_col = f"n_author_{topic_col.replace('primary_for_', '')}_topics"
        share_col = f"modal_{topic_col}_share"

        counts = (author_papers.groupby(["author_id", topic_col])
                  .size().reset_index(name=count_col))
        totals = counts.groupby("author_id")[count_col].sum().rename(total_col)
        n_topics = counts.groupby("author_id")[topic_col].nunique().rename(n_col)
        counts["is_unclassified"] = counts[topic_col].eq("Unclassified")
        counts = counts.sort_values(
            ["author_id", count_col, "is_unclassified", topic_col],
            ascending=[True, False, True, True],
        )
        modal = counts.drop_duplicates("author_id", keep="first").drop(columns="is_unclassified")
        modal = modal.rename(columns={topic_col: f"modal_{topic_col}"})
        modal = modal.merge(totals, on="author_id", how="left").merge(n_topics, on="author_id", how="left")
        modal[share_col] = modal[count_col] / modal[total_col]
        return modal

    modal_l2 = modal_for("primary_for_l2")
    modal_l4 = modal_for("primary_for_l4")
    modal = modal_l2.merge(modal_l4, on="author_id", how="outer")

    # Backward-compatible aliases for the original L2 fields.
    modal["modal_topic_papers"] = modal["modal_primary_for_l2_papers"]
    modal["topic_observed_papers"] = modal["primary_for_l2_observed_papers"]
    modal["n_author_topics"] = modal["n_author_l2_topics"]
    modal["modal_topic_share"] = modal["modal_primary_for_l2_share"]
    return modal.sort_values("author_id").reset_index(drop=True)


def get_author_metadata_source(df_all=None, path="../data/df_dimensions.xlsx"):
    """Return paper IDs and authors metadata, loading it if needed."""
    if df_all is not None and {"id", "authors"}.issubset(df_all.columns):
        return df_all[["id", "authors"]].copy()
    return pd.read_excel(path, usecols=["id", "authors"])


def pick_main_affiliation_for_community(author):
    """Pick one institution for an author on a paper, matching 01_analysis_03_authors."""
    affiliations = author.get("affiliations") or []
    if not affiliations:
        return {}, "none"
    current_org_id = author.get("current_organization_id")
    if current_org_id:
        for affiliation in affiliations:
            if isinstance(affiliation, dict) and affiliation.get("id") == current_org_id:
                return affiliation, "current_org"
    first = affiliations[0]
    return (first if isinstance(first, dict) else {}), "first_listed"


def make_author_modal_institution_table(df_year, author_metadata_df):
    """Assign each author their modal main institution over papers in df_year."""
    paper_ids = set(df_year["id"])
    source = author_metadata_df[author_metadata_df["id"].isin(paper_ids)].copy()
    records = []
    for _, row in source.iterrows():
        paper_id = row["id"]
        for author in parse_listcol_for_topics(row["authors"]):
            if not isinstance(author, dict):
                continue
            author_id = author.get("researcher_id") or author.get("id")
            if not author_id:
                continue
            affiliation, source_type = pick_main_affiliation_for_community(author)
            records.append({
                "author_id": author_id,
                "paper_id": paper_id,
                "institution": affiliation.get("name") or "Unknown institution",
                "institution_id": affiliation.get("id"),
                "institution_country": affiliation.get("country"),
                "institution_source": source_type,
            })

    author_institutions = pd.DataFrame(records).drop_duplicates(["author_id", "paper_id"])
    if author_institutions.empty:
        return pd.DataFrame(columns=[
            "author_id", "modal_institution", "modal_institution_papers",
            "institution_observed_papers", "modal_institution_share",
        ])

    counts = (author_institutions.groupby(["author_id", "institution"])
              .size().reset_index(name="modal_institution_papers"))
    totals = counts.groupby("author_id")["modal_institution_papers"].sum().rename("institution_observed_papers")
    counts["is_unknown"] = counts["institution"].eq("Unknown institution")
    counts = counts.sort_values(
        ["author_id", "modal_institution_papers", "is_unknown", "institution"],
        ascending=[True, False, True, True],
    )
    modal = counts.drop_duplicates("author_id", keep="first").drop(columns="is_unknown")
    modal = modal.rename(columns={"institution": "modal_institution"})
    modal = modal.merge(totals, on="author_id", how="left")
    modal["modal_institution_share"] = modal["modal_institution_papers"] / modal["institution_observed_papers"]
    return modal.sort_values("author_id").reset_index(drop=True)


def make_topic_summary_by_group(author_topics_df, group_col, topic_col="modal_primary_for_l2", top_n=3):
    """Summarise the dominant and top-n modal author topics for each group."""
    base = author_topics_df[[group_col, "author_id", topic_col]].copy()
    base[topic_col] = base[topic_col].fillna("Unclassified")
    counts = (base.groupby([group_col, topic_col])["author_id"]
              .nunique().reset_index(name="n_authors"))
    totals = base.groupby(group_col)["author_id"].nunique().rename("group_authors")
    counts = counts.merge(totals, on=group_col, how="left")
    counts["topic_share"] = counts["n_authors"] / counts["group_authors"]
    counts["is_unclassified"] = counts[topic_col].eq("Unclassified")
    counts = counts.sort_values(
        [group_col, "n_authors", "is_unclassified", topic_col],
        ascending=[True, False, True, True],
    )
    counts["topic_rank"] = counts.groupby(group_col).cumcount() + 1

    dominant = counts[counts["topic_rank"] == 1].copy()
    dominant = dominant.rename(columns={
        topic_col: "dominant_topic",
        "n_authors": "dominant_topic_authors",
        "topic_share": "dominant_topic_share",
    })[[group_col, "dominant_topic", "dominant_topic_authors", "dominant_topic_share", "group_authors"]]

    top_topics = (counts[counts["topic_rank"] <= top_n]
                  .assign(topic_text=lambda d: d[topic_col] + " (" +
                          d["n_authors"].astype(str) + ", " +
                          (100 * d["topic_share"]).round(1).astype(str) + "%)")
                  .groupby(group_col)["topic_text"]
                  .apply(lambda values: "; ".join(values))
                  .rename("top_topics"))

    summary = dominant.merge(top_topics, on=group_col, how="left")
    return summary, counts.drop(columns="is_unclassified")


def make_institution_summary_by_group(author_df, group_col, institution_col="modal_institution", top_n=3):
    """Summarise the top institutions represented by authors in each group."""
    base = author_df[[group_col, "author_id", institution_col]].copy()
    base[institution_col] = base[institution_col].fillna("Unknown institution")
    counts = (base.groupby([group_col, institution_col])["author_id"]
              .nunique().reset_index(name="n_authors"))
    totals = base.groupby(group_col)["author_id"].nunique().rename("group_authors")
    counts = counts.merge(totals, on=group_col, how="left")
    counts["institution_share"] = counts["n_authors"] / counts["group_authors"]
    counts["is_unknown"] = counts[institution_col].eq("Unknown institution")
    counts = counts.sort_values(
        [group_col, "n_authors", "is_unknown", institution_col],
        ascending=[True, False, True, True],
    )
    counts["institution_rank"] = counts.groupby(group_col).cumcount() + 1

    top = counts[counts["institution_rank"] == 1].copy()
    top = top.rename(columns={
        institution_col: "top_institution",
        "n_authors": "top_institution_authors",
    })[[group_col, "top_institution", "top_institution_authors", "institution_share", "group_authors"]]
    top = top.rename(columns={"institution_share": "top_institution_share"})

    top_institutions = (counts[counts["institution_rank"] <= top_n]
                        .assign(institution_text=lambda d: d[institution_col] + " (" +
                                d["n_authors"].astype(str) + ", " +
                                (100 * d["institution_share"]).round(1).astype(str) + "%)")
                        .groupby(group_col)["institution_text"]
                        .apply(lambda values: "; ".join(values))
                        .rename("top_institutions"))

    summary = top.merge(top_institutions, on=group_col, how="left")
    return summary, counts.drop(columns="is_unknown")


def short_topic_label(topic, width=34):
    """Compact long FOR labels for in-figure community labels."""
    if not isinstance(topic, str) or not topic.strip():
        return "Unclassified"
    label = TOPIC_LABEL_OVERRIDES.get(topic.strip(), topic.strip())
    return textwrap.shorten(label, width=width, placeholder="...")


def short_institution_label(institution, width=30):
    """Compact institution names for in-figure labels."""
    if not isinstance(institution, str) or not institution.strip():
        return "Unknown institution"
    label = INSTITUTION_LABEL_OVERRIDES.get(institution.strip(), institution.strip())
    return textwrap.shorten(label, width=width, placeholder="...")


def make_display_topic_label(topic_counts, topic_col="modal_primary_for_l4", width=34, mixed_share_threshold=0.55):
    """Use a top-two topic mix when no single modal topic clearly dominates."""
    if topic_counts is None or len(topic_counts) == 0:
        return "Unclassified"
    rows = topic_counts.sort_values("topic_rank")
    top = rows.iloc[0]
    top_label = short_topic_label(top[topic_col], width=min(width, 24))
    if len(rows) > 1 and float(top["topic_share"]) < mixed_share_threshold:
        second = rows.iloc[1]
        second_label = short_topic_label(second[topic_col], width=min(width, 24))
        return f"{top_label} + {second_label}"
    return short_topic_label(top[topic_col], width=width)


def apply_topic_labels_to_plot_graph(
    plot_graph,
    community_topic_summary,
    other_label,
    width=34,
    community_topic_counts=None,
    community_institution_summary=None,
    topic_col="modal_primary_for_l4",
    mixed_share_threshold=0.55,
):
    """Use FOR topic and top institution labels for retained communities."""
    topic_lookup = community_topic_summary.set_index("leiden_community").to_dict("index")
    institution_lookup = {}
    if community_institution_summary is not None and not community_institution_summary.empty:
        institution_lookup = community_institution_summary.set_index("leiden_community").to_dict("index")
    counts_lookup = {}
    if community_topic_counts is not None and not community_topic_counts.empty:
        counts_lookup = {
            int(community_id): rows.copy()
            for community_id, rows in community_topic_counts.groupby("leiden_community")
        }
    for node, data in plot_graph.nodes(data=True):
        if data.get("is_other"):
            data["display_label"] = other_label
            data["display_topic_label"] = other_label
            data["dominant_topic"] = pd.NA
            data["dominant_topic_share"] = pd.NA
            data["top_institution"] = pd.NA
            data["top_institution_share"] = pd.NA
            continue
        community_id = int(data["original_communities"][0])
        topic_info = topic_lookup.get(community_id, {})
        topic = topic_info.get("dominant_topic", "Unclassified")
        display_topic_label = make_display_topic_label(
            counts_lookup.get(community_id),
            topic_col=topic_col,
            width=width,
            mixed_share_threshold=mixed_share_threshold,
        )
        institution_info = institution_lookup.get(community_id, {})
        top_institution = institution_info.get("top_institution", "Unknown institution")
        institution_label = short_institution_label(top_institution)
        data["dominant_topic"] = topic
        data["dominant_topic_share"] = topic_info.get("dominant_topic_share", pd.NA)
        data["top_institution"] = top_institution
        data["top_institution_share"] = institution_info.get("top_institution_share", pd.NA)
        data["display_topic_label"] = display_topic_label
        data["display_institution_label"] = institution_label
        data["display_label"] = f"C{int(data['rank'])}: {display_topic_label}\n{institution_label}"
    return plot_graph


def build_full_community_graph(author_graph, membership_by_node, weight_attr="weight"):
    """
    Aggregate the author graph to the full Leiden community graph.

    Nodes are Leiden communities with size = number of authors. Edge weights are
    the total number of co-authorship collaborations between communities.
    """
    community_sizes = Counter(membership_by_node.values())
    community_graph = nx.Graph()

    for community_id, size in community_sizes.items():
        community_graph.add_node(
            int(community_id),
            size=int(size),
            label=f"Leiden {community_id}",
            internal_weight=0.0,
        )

    for u, v, data in author_graph.edges(data=True):
        community_u = int(membership_by_node[u])
        community_v = int(membership_by_node[v])
        weight = float(data.get(weight_attr, 1.0))

        if community_u == community_v:
            community_graph.nodes[community_u]["internal_weight"] += weight
            continue

        if community_graph.has_edge(community_u, community_v):
            community_graph[community_u][community_v]["weight"] += weight
        else:
            community_graph.add_edge(community_u, community_v, weight=weight)

    return community_graph


def collapse_community_graph(
    full_community_graph,
    n_largest=25,
    other_label="Other Communities",
    omit_smaller_than=None,
):
    """
    Keep the largest Leiden communities as separate display nodes and merge all
    remaining communities into one Other Communities node.
    """
    size_series = pd.Series(
        nx.get_node_attributes(full_community_graph, "size"),
        name="n_authors",
    ).sort_values(ascending=False)

    rank_by_community = {
        int(community_id): rank
        for rank, community_id in enumerate(size_series.index.tolist(), start=1)
    }
    kept_communities = set(int(c) for c in size_series.head(n_largest).index)

    community_to_plot_node = {}
    for community_id, size in size_series.items():
        community_id = int(community_id)
        if omit_smaller_than is not None and size < omit_smaller_than:
            community_to_plot_node[community_id] = None
        elif community_id in kept_communities:
            community_to_plot_node[community_id] = f"Community {rank_by_community[community_id]}"
        else:
            community_to_plot_node[community_id] = other_label

    plot_graph = nx.Graph()

    for community_id, size in size_series.items():
        community_id = int(community_id)
        plot_node = community_to_plot_node[community_id]
        if plot_node is None:
            continue

        if plot_node not in plot_graph:
            is_other = plot_node == other_label
            plot_graph.add_node(
                plot_node,
                size=0,
                display_label=other_label if is_other else plot_node,
                rank=None if is_other else rank_by_community[community_id],
                is_other=is_other,
                n_leiden_communities=0,
                original_communities=[],
                internal_weight=0.0,
                internal_merged_weight=0.0,
            )

        node_data = plot_graph.nodes[plot_node]
        node_data["size"] += int(size)
        node_data["n_leiden_communities"] += 1
        node_data["original_communities"].append(community_id)
        node_data["internal_weight"] += float(
            full_community_graph.nodes[community_id].get("internal_weight", 0.0)
        )

    for community_u, community_v, data in full_community_graph.edges(data=True):
        plot_u = community_to_plot_node[int(community_u)]
        plot_v = community_to_plot_node[int(community_v)]
        if plot_u is None or plot_v is None:
            continue

        weight = float(data.get("weight", 1.0))
        if plot_u == plot_v:
            plot_graph.nodes[plot_u]["internal_merged_weight"] += weight
            continue

        if plot_graph.has_edge(plot_u, plot_v):
            plot_graph[plot_u][plot_v]["weight"] += weight
        else:
            plot_graph.add_edge(plot_u, plot_v, weight=weight)

    community_summary = pd.DataFrame({
        "leiden_community": [int(c) for c in size_series.index],
        "rank": [rank_by_community[int(c)] for c in size_series.index],
        "n_authors": [int(size_series.loc[c]) for c in size_series.index],
        "plot_node": [community_to_plot_node[int(c)] for c in size_series.index],
    })
    community_summary["retained_as_separate_node"] = community_summary["plot_node"].str.startswith("Community ", na=False)
    community_summary["merged_into_other"] = community_summary["plot_node"].eq(other_label)
    community_summary["omitted_from_plot"] = community_summary["plot_node"].isna()

    return plot_graph, community_summary, community_to_plot_node


In [ ]:
# Build the 2025 author graph, detect Leiden communities, and create outputs.
if "df_clean" not in globals():
    df_clean = df[df["year"].notna()].copy()
    df_clean["year"] = df_clean["year"].astype(int)
    df_clean = df_clean.sort_values("year")

G_2025, author_mapping_2025, df_2025_slice, leiden_slice_label = build_author_graph_for_analysis_year(
    df_clean,
    analysis_year=ANALYSIS_YEAR,
    cumulative=USE_CUMULATIVE_NETWORK_THROUGH_YEAR,
)

print(f"Author graph for Leiden analysis: {leiden_slice_label}")
print(f"Papers included: {len(df_2025_slice):,}")
print(f"Authors: {G_2025.number_of_nodes():,}")
print(f"Author-author collaboration edges: {G_2025.number_of_edges():,}")

membership_by_author_node, leiden_partition, igraph_2025, leiden_modularity = detect_leiden_communities(
    G_2025,
    resolution=LEIDEN_RESOLUTION,
    seed=RANDOM_SEED,
)

author_communities_df = make_author_community_table(
    membership_by_author_node,
    author_mapping_2025,
)

paper_topic_source_df = get_paper_topic_source(globals().get("df_all"))
paper_topics_df = build_paper_topic_table(paper_topic_source_df)
author_modal_topics_df = make_author_modal_topic_table(df_2025_slice, paper_topics_df)
author_communities_df = author_communities_df.merge(
    author_modal_topics_df,
    on="author_id",
    how="left",
)
author_communities_df["modal_primary_for_l2"] = author_communities_df["modal_primary_for_l2"].fillna("Unclassified")
author_communities_df["modal_topic_papers"] = author_communities_df["modal_topic_papers"].fillna(0).astype(int)
author_communities_df["topic_observed_papers"] = author_communities_df["topic_observed_papers"].fillna(0).astype(int)
author_communities_df["n_author_topics"] = author_communities_df["n_author_topics"].fillna(0).astype(int)

paper_topics_path = COMMUNITY_OUTPUT_DIR / f"paper_primary_for_l2_{ANALYSIS_YEAR}.csv"
author_modal_topics_path = COMMUNITY_OUTPUT_DIR / f"author_modal_topics_{ANALYSIS_YEAR}.csv"
author_communities_path = COMMUNITY_OUTPUT_DIR / f"author_leiden_membership_{ANALYSIS_YEAR}.csv"
paper_topics_df.to_csv(paper_topics_path, index=False)
author_modal_topics_df.to_csv(author_modal_topics_path, index=False)
author_communities_df.to_csv(author_communities_path, index=False)

full_community_graph = build_full_community_graph(G_2025, membership_by_author_node)
plot_community_graph, community_summary_df, community_to_plot_node = collapse_community_graph(
    full_community_graph,
    n_largest=N_LARGEST_COMMUNITIES_TO_KEEP,
    other_label=OTHER_COMMUNITIES_LABEL,
    omit_smaller_than=OMIT_COMMUNITIES_SMALLER_THAN,
)

community_topic_summary_df, community_topic_counts_df = make_topic_summary_by_group(
    author_communities_df,
    group_col="leiden_community",
    top_n=3,
)
community_summary_df = community_summary_df.merge(
    community_topic_summary_df,
    on="leiden_community",
    how="left",
)
plot_community_graph = apply_topic_labels_to_plot_graph(
    plot_community_graph,
    community_topic_summary_df,
    other_label=OTHER_COMMUNITIES_LABEL,
    community_topic_counts=community_topic_counts_df,
    mixed_share_threshold=MIXED_TOPIC_LABEL_SHARE_THRESHOLD,
)

author_plot_topics_df = author_communities_df.copy()
author_plot_topics_df["plot_node"] = author_plot_topics_df["leiden_community"].map(community_to_plot_node)
author_plot_topics_df = author_plot_topics_df.dropna(subset=["plot_node"])
plot_topic_summary_df, plot_topic_counts_df = make_topic_summary_by_group(
    author_plot_topics_df,
    group_col="plot_node",
    top_n=5,
)

community_summary_path = COMMUNITY_OUTPUT_DIR / f"leiden_community_summary_{ANALYSIS_YEAR}.csv"
community_topic_summary_path = COMMUNITY_OUTPUT_DIR / f"leiden_community_topic_summary_{ANALYSIS_YEAR}.csv"
community_topic_counts_path = COMMUNITY_OUTPUT_DIR / f"leiden_community_topic_counts_{ANALYSIS_YEAR}.csv"
plot_topic_summary_path = COMMUNITY_OUTPUT_DIR / f"collapsed_community_topic_summary_{ANALYSIS_YEAR}.csv"
plot_topic_counts_path = COMMUNITY_OUTPUT_DIR / f"collapsed_community_topic_counts_{ANALYSIS_YEAR}.csv"
community_summary_df.to_csv(community_summary_path, index=False)
community_topic_summary_df.to_csv(community_topic_summary_path, index=False)
community_topic_counts_df.to_csv(community_topic_counts_path, index=False)
plot_topic_summary_df.to_csv(plot_topic_summary_path, index=False)
plot_topic_counts_df.to_csv(plot_topic_counts_path, index=False)

plot_nodes_df = pd.DataFrame([
    {
        "plot_node": node,
        "label": data["display_label"],
        "display_topic_label": data.get("display_topic_label"),
        "n_authors": int(data["size"]),
        "rank": data["rank"],
        "is_other": data["is_other"],
        "dominant_topic": data.get("dominant_topic"),
        "dominant_topic_share": data.get("dominant_topic_share"),
        "n_leiden_communities": int(data["n_leiden_communities"]),
        "original_communities": ";".join(map(str, data["original_communities"])),
        "within_display_node_weight": data["internal_weight"] + data["internal_merged_weight"],
    }
    for node, data in plot_community_graph.nodes(data=True)
]).sort_values(["is_other", "rank"], na_position="last")
plot_nodes_df = plot_nodes_df.merge(
    plot_topic_summary_df[["plot_node", "dominant_topic", "dominant_topic_share", "top_topics"]].rename(columns={
        "dominant_topic": "collapsed_dominant_topic",
        "dominant_topic_share": "collapsed_dominant_topic_share",
        "top_topics": "collapsed_top_topics",
    }),
    on="plot_node",
    how="left",
)

plot_edges_df = pd.DataFrame([
    {
        "source": u,
        "target": v,
        "weight": float(data.get("weight", 1.0)),
    }
    for u, v, data in plot_community_graph.edges(data=True)
]).sort_values("weight", ascending=False, ignore_index=True)

plot_nodes_path = COMMUNITY_OUTPUT_DIR / f"collapsed_community_nodes_{ANALYSIS_YEAR}.csv"
plot_edges_path = COMMUNITY_OUTPUT_DIR / f"collapsed_community_edges_{ANALYSIS_YEAR}.csv"
plot_nodes_df.to_csv(plot_nodes_path, index=False)
plot_edges_df.to_csv(plot_edges_path, index=False)

community_sizes = community_summary_df["n_authors"].sort_values(ascending=False).reset_index(drop=True)
community_size_distribution = community_sizes.describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)

print("\n" + "=" * 72)
print(f"Leiden community statistics ({leiden_slice_label})")
print("=" * 72)
print(f"Number of Leiden communities: {len(community_sizes):,}")
print(f"Size of largest community: {int(community_sizes.iloc[0]):,} authors")
print(f"Network modularity: {leiden_modularity:.4f}")
print(f"Number of inter-community edges: {full_community_graph.number_of_edges():,}")
print(f"Collapsed display nodes: {plot_community_graph.number_of_nodes():,}")
print(f"Collapsed display edges: {plot_community_graph.number_of_edges():,}")
print("\nCommunity size distribution:")
print(community_size_distribution.to_string(float_format=lambda value: f"{value:,.2f}"))
print("\nLargest Leiden communities:")
largest_display_cols = [
    "rank", "n_authors", "plot_node", "dominant_topic",
    "dominant_topic_share", "top_topics",
]
print(community_summary_df[largest_display_cols].head(15).to_string(index=False))
print("\nSaved data outputs:")
print(f"- {paper_topics_path}")
print(f"- {author_modal_topics_path}")
print(f"- {author_communities_path}")
print(f"- {community_summary_path}")
print(f"- {community_topic_summary_path}")
print(f"- {community_topic_counts_path}")
print(f"- {plot_topic_summary_path}")
print(f"- {plot_topic_counts_path}")
print(f"- {plot_nodes_path}")
print(f"- {plot_edges_path}")


In [ ]:
def make_value_scaler(values, out_min, out_max, transform=None):
    """Return a callable that maps data values into a plotting range."""
    values = np.asarray([float(v) for v in values if pd.notna(v)], dtype=float)
    if values.size == 0:
        return lambda value: (out_min + out_max) / 2

    transformed = transform(values) if transform is not None else values
    v_min = float(np.min(transformed))
    v_max = float(np.max(transformed))

    if math.isclose(v_min, v_max):
        return lambda value: (out_min + out_max) / 2

    def scaler(value):
        transformed_value = transform(float(value)) if transform is not None else float(value)
        return out_min + (transformed_value - v_min) * (out_max - out_min) / (v_max - v_min)

    return scaler


def legend_values(values):
    """Pick compact min/median/max legend values from observed data."""
    arr = np.asarray([float(v) for v in values if float(v) > 0], dtype=float)
    if arr.size == 0:
        return []
    picked = [np.min(arr), np.median(arr), np.max(arr)]
    return sorted({int(round(v)) for v in picked})


def reduce_node_overlap(pos, node_areas, iterations=250, padding=0.035):
    """
    Light collision pass for the small collapsed graph. This keeps the spring
    layout structure but nudges nodes apart when large symbols overlap.
    """
    nodes = list(pos.keys())
    if len(nodes) < 2:
        return pos

    coords = np.asarray([pos[node] for node in nodes], dtype=float)
    max_area = max(float(node_areas.get(node, 1.0)) for node in nodes)
    radii = np.asarray([
        0.11 * math.sqrt(float(node_areas.get(node, 1.0)) / max_area)
        for node in nodes
    ])

    for _ in range(iterations):
        moved = False
        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                delta = coords[j] - coords[i]
                distance = float(np.linalg.norm(delta))
                if distance == 0:
                    angle = (i + j + 1) * math.pi / 7
                    direction = np.asarray([math.cos(angle), math.sin(angle)])
                    distance = 1e-9
                else:
                    direction = delta / distance

                min_distance = radii[i] + radii[j] + padding
                if distance < min_distance:
                    shift = 0.5 * (min_distance - distance) * direction
                    coords[i] -= shift
                    coords[j] += shift
                    moved = True

        coords -= coords.mean(axis=0)
        if not moved:
            break

    max_abs = float(np.max(np.abs(coords)))
    if max_abs > 0:
        coords = coords / max_abs

    return {node: coords[idx] for idx, node in enumerate(nodes)}


def plot_leiden_community_network(
    plot_graph,
    modularity,
    slice_label,
    output_stem,
    label_top_n=10,
    label_min_authors=500,
    seed=48652,
    resolution=1.0,
):
    """Create and save a publication-quality collapsed community network."""
    graph = plot_graph.copy()
    if graph.number_of_nodes() == 0:
        raise ValueError("The collapsed community graph has no nodes to plot.")

    for u, v, data in graph.edges(data=True):
        data["layout_weight"] = math.log1p(float(data.get("weight", 1.0)))

    sizes = {node: int(data["size"]) for node, data in graph.nodes(data=True)}
    edge_weights = [float(data.get("weight", 1.0)) for _, _, data in graph.edges(data=True)]

    node_size_scaler = make_value_scaler(sizes.values(), 450, 4300, transform=np.sqrt)
    edge_width_scaler = make_value_scaler(edge_weights, 0.35, 5.0, transform=np.log1p)
    node_areas = {node: node_size_scaler(size) for node, size in sizes.items()}

    layout_k = 1.35 / math.sqrt(max(graph.number_of_nodes(), 1))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        pos = nx.spring_layout(
            graph,
            weight="layout_weight",
            k=layout_k,
            iterations=900,
            seed=seed,
            scale=1.0,
        )
    pos = reduce_node_overlap(pos, node_areas)

    palette = list(plt.get_cmap("tab20").colors) + list(plt.get_cmap("tab20b").colors)
    node_colors = []
    for node, data in graph.nodes(data=True):
        if data.get("is_other"):
            node_colors.append("#B8B8B8")
        else:
            rank = int(data.get("rank") or 1)
            node_colors.append(palette[(rank - 1) % len(palette)])

    fig, ax = plt.subplots(figsize=(11.5, 9.0), dpi=300)
    ax.set_facecolor("white")

    edge_widths = [edge_width_scaler(weight) for weight in edge_weights]
    nx.draw_networkx_edges(
        graph,
        pos,
        ax=ax,
        width=edge_widths,
        edge_color="#4A4A4A",
        alpha=0.28,
    )

    nx.draw_networkx_nodes(
        graph,
        pos,
        ax=ax,
        node_size=[node_areas[node] for node in graph.nodes()],
        node_color=node_colors,
        edgecolors="#202020",
        linewidths=0.8,
        alpha=0.92,
    )

    ranked_nodes = sorted(
        graph.nodes(data=True),
        key=lambda item: item[1].get("size", 0),
        reverse=True,
    )
    label_nodes = {
        node
        for node, data in ranked_nodes[:label_top_n]
        if int(data.get("size", 0)) >= label_min_authors or data.get("is_other")
    }
    label_nodes.update(
        node for node, data in graph.nodes(data=True) if data.get("is_other")
    )

    for node in label_nodes:
        x, y = pos[node]
        data = graph.nodes[node]
        label = f"{data['display_label']}\n{int(data['size']):,} authors"
        ax.text(
            x,
            y,
            label,
            ha="center",
            va="center",
            fontsize=8.5 if not data.get("is_other") else 8.0,
            color="#111111",
            linespacing=1.15,
            path_effects=[pe.withStroke(linewidth=3.5, foreground="white")],
        )

    ax.set_title(
        f"UK Biobank co-authorship communities ({slice_label})",
        loc="left",
        fontsize=16,
        fontweight="bold",
        pad=22,
    )
    ax.text(
        0.0,
        0.995,
        (
            f"Leiden resolution {resolution:g}; modularity {modularity:.3f}; "
            f"{graph.number_of_nodes()} displayed nodes; {graph.number_of_edges()} displayed inter-community edges"
        ),
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10.5,
        color="#333333",
    )

    node_legend_marker_scaler = make_value_scaler(sizes.values(), 7, 22, transform=np.sqrt)
    node_handles = [
        mlines.Line2D(
            [],
            [],
            marker="o",
            linestyle="None",
            markersize=node_legend_marker_scaler(value),
            markerfacecolor="#8AA7D6",
            markeredgecolor="#202020",
            markeredgewidth=0.8,
            alpha=0.92,
            label=f"{value:,} authors",
        )
        for value in legend_values(sizes.values())
    ]
    edge_handles = [
        mlines.Line2D(
            [],
            [],
            color="#4A4A4A",
            linewidth=edge_width_scaler(value),
            alpha=0.45,
            label=f"{value:,} collaborations",
        )
        for value in legend_values(edge_weights)
    ]

    if node_handles:
        legend_nodes = ax.legend(
            handles=node_handles,
            title="Node size",
            loc="lower left",
            frameon=True,
            framealpha=0.96,
            facecolor="white",
            edgecolor="#DDDDDD",
            fontsize=8.5,
            title_fontsize=9.5,
            borderpad=0.8,
            labelspacing=0.9,
        )
        ax.add_artist(legend_nodes)

    if edge_handles:
        ax.legend(
            handles=edge_handles,
            title="Edge weight",
            loc="lower right",
            frameon=True,
            framealpha=0.96,
            facecolor="white",
            edgecolor="#DDDDDD",
            fontsize=8.5,
            title_fontsize=9.5,
            borderpad=0.8,
            labelspacing=0.9,
        )

    ax.set_axis_off()
    ax.margins(0.16)
    fig.tight_layout(rect=[0, 0, 1, 0.97])

    output_paths = {
        "png": output_stem.with_suffix(".png"),
        "svg": output_stem.with_suffix(".svg"),
        "pdf": output_stem.with_suffix(".pdf"),
    }
    for output_path in output_paths.values():
        fig.savefig(output_path, bbox_inches="tight", facecolor="white", dpi=400)

    return fig, ax, pos, output_paths


community_figure_stem = FIG_OUTPUT_DIR / f"ukb_leiden_community_network_{ANALYSIS_YEAR}"
fig, ax, community_layout_pos, community_figure_paths = plot_leiden_community_network(
    plot_community_graph,
    modularity=leiden_modularity,
    slice_label=leiden_slice_label,
    output_stem=community_figure_stem,
    label_top_n=LABEL_TOP_N_COMMUNITIES,
    label_min_authors=LABEL_MIN_AUTHORS,
    seed=RANDOM_SEED,
    resolution=LEIDEN_RESOLUTION,
)

print("Saved figure outputs:")
for fmt, path in community_figure_paths.items():
    print(f"- {fmt.upper()}: {path}")

try:
    get_ipython
except NameError:
    plt.close(fig)
else:
    plt.show()
